# MODIS Terra L1B Extraction

This notebook demonstrates how to search for MODIS Terra L1B granules via **earthaccess** and extract thermal bands using **satpy**.

## Setup

Import the required libraries and configure the search parameters.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile
from aer.search_earthaccess import earthaccess_download_wrapper

## Configuration

Define the AOI, time range, and collection.

In [ ]:
# --- Configuration ---
DATE_START = datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc)
DATE_END = datetime(2026, 4, 5, 0, 0, tzinfo=timezone.utc)

# Load AOI (Buenos Aires province)
geojson_path = Path("buenos_aires.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# MODIS Terra L1B 1km collection
collections = ["MOD021KM"]

# --- Client Setup ---
client = AerClient()

## Search

Search for MODIS granules that intersect the AOI using `search_earthaccess`.

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=collections,
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
    plugin_hints={"search_earthaccess": collections},
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Prepare Extraction

Define extraction profiles and prepare tasks.

In [ ]:
# --- Prepare Extraction ---
uri = "extract_buenos_aires_modis"

profiles = [
    ExtractionProfile(
        name="modis_thermal",
        resolution=1000,
        collection_variables_map={"MOD021KM": ["31"]},
        extra_params={"reader": "modis_l1b"},
    )
]

prepare_params = {"cells_per_chunk": 10}

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params=prepare_params,
    plugin_hints={"extract_satpy": collections},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

Run the extraction pipeline.

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

extract_params = {
    "downloader": earthaccess_download_wrapper,
    "calibration": "brightness_temperature",
    "padding": 2,
    "resampler": "nearest",
}

results_df = client.extract_batches(
    tasks,
    extract_params=extract_params,
    plugin_hints={"extract_satpy": collections},
    max_batch_workers=4,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

Inspect the extracted artifacts.

In [ ]:
results_df[["id", "collection", "grid_cell", "uri"]].head()